In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Keys loaded:", PINECONE_API_KEY is not None, GROQ_API_KEY is not None)

Keys loaded: True True


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings loaded!")

d:\MISSION X\Medical-QA\medibot_win\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Embeddings loaded!


In [3]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name="medical-qa",
    embedding=embedding
)

retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Pinecone connected!")

Pinecone connected!


In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("LLM loaded!")

LLM loaded!


In [10]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a helpful medical assistant. "
    "Use the following retrieved context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Keep your answer concise — three sentences maximum.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answering_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Acne vulgaris, also known as common acne, is the most common skin disease, affecting nearly 17 million people in the United States.


In [13]:
response = rag_chain.invoke({"input": "Tell me about Diabetes?How can i prevent it?"})
print(response["answer"])

Diabetes is a condition where the body either doesn't make enough insulin or the insulin doesn't work properly, resulting in high blood sugar levels. To prevent diabetes, it's recommended to maintain a healthy weight, engage in regular physical activity, and eat a balanced diet low in sugar and saturated fats. Additionally, getting regular check-ups and screenings can help identify risk factors and detect diabetes early, allowing for prompt treatment and management.
